## IMPORT

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import sklearn
from datetime import datetime
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import joblib
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.multiclass import OneVsRestClassifier

## Modèle 1 : Prédiction de la taille d'un panier pour un acheteur d'alcool

In [8]:
df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")

#/-------------------------------------------------------\#
#|------------- Tri et création de colonnes -------------|#
#\-------------------------------------------------------/#

# Récupère si l'article dans le panier est de l'alcool
df["is_alcohol"] = df["department"].str.contains("alcohol", case=False, na=False).astype(int)

# Création d'un nouveau dataset en récupérant la taille de la commande des clients en fonction de l'id de la commande
basket_size = df.groupby("order_id").size().reset_index(name="basket_size")

# Création d'un nouveau dataset en récupérant les données en fonction de l'id de la commande qui me permettra de trier plus tard
orders = df.groupby("order_id").agg(
    user_id=("user_id", "first"),
    order_number=("order_number", "first"),
    order_hour_of_day=("order_hour_of_day", "first"),
    order_dow=("order_dow", "first"),
    days_since_prior_order=("days_since_prior_order", "first"),
    alcohol_count=("is_alcohol", "sum"),
    reorder_count=("reordered", "sum"),
).reset_index()

# Ce merge là sert à trier une nouvelle fois en fusionnant avec le datasets basket_size en fonction de l'id de la commande
orders = orders.merge(basket_size, on="order_id")

# Récupère le nombre d'article d'alcool en fonction de la taille du panier
orders["alcohol_ratio"] = orders["alcohol_count"]/orders["basket_size"]

# Récupère le nombre d'article commendés plusieurs fois en fonction de la taille du panier
orders["reorder_ratio"]  = orders["reorder_count"]/orders["basket_size"]

# Récupère si on est le week-end en récupérant à partir de la colonnes order_dow en prennant uniquement les valeurs entre 6 et 0 (donc Samedi et Dimanche)
orders["is_weekend"] = orders["order_dow"].isin([6, 0]).astype(int)
orders["time_of_day"] = pd.cut(
    orders["order_hour_of_day"],
    bins=[0, 6, 12, 17, 21],
    labels=["nuit", "matin", "apres_midi", "soir"],
    right=False
).astype(str)

#/--------------------------------------------------------------\#
#|------------------------ Entraînement ------------------------|#
#\--------------------------------------------------------------/#

# Ici je définis les colonnes que le modèle doit utiliser pour s'entraîner
feature_cols = [
    "order_hour_of_day", "order_dow",
    "is_weekend", "time_of_day",
    "days_since_prior_order", "order_number",
    "alcohol_count", "alcohol_ratio",
    "reorder_count", "reorder_ratio",
]

# Transforme les chaînes de charactère en True/False et définis mon X
X = pd.get_dummies(orders[feature_cols])

# Récupère le y, la target
y = orders["basket_size"]

# Split les données entre tests et entraînements 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=42
)

# Définis le modèle utiliser et ses hyperparamètres
model = HistGradientBoostingRegressor(
    max_iter=500,                 
    learning_rate=0.05,
    max_leaf_nodes=127,            
    min_samples_leaf=50,                    
    random_state=42,
)

# Lance l'entraînements du modèle
model.fit(X_train, y_train)

#/--------------------------------------------------------------\#
#|------------------------- Évaluation -------------------------|#
#\--------------------------------------------------------------/#

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE :{mae}")
print(f"RMSE :{rmse}")
print(f"R2 :{r2}")
print(orders)

MAE :0.20215178100985065
RMSE :0.919412535420105
R2 :0.983796033671207
         order_id  user_id  order_number  order_hour_of_day  order_dow  \
0               2   202279             3                  9          5   
1               4   178520            36                  9          1   
2               6    22352             4                 12          1   
3               8     3107             5                  6          4   
4              21    88772            13                 12          2   
...           ...      ...           ...                ...        ...   
1270537   2999995   128711             4                 14          5   
1270538   2999996   120666             5                 14          3   
1270539   2999997    41179            13                 13          1   
1270540   2999998    68878             2                 13          2   
1270541   2999999   100467             5                  8          2   

         days_since_prior_order  alcohol

## Modèle 2 : Prédiction du contenu du panier

In [17]:
df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")

top_500_products = df["product_name"].value_counts().head(500).index
df_filtered = df[df["product_name"].isin(top_500_products)]

grouped_by_orders = df_filtered.groupby("order_id").agg({
    "product_name": list,
    "user_id": "first",
    "order_number": "first",
    "order_dow": "first",
    "order_hour_of_day": "first",
    "days_since_prior_order": "first",
    
}).reset_index()

grouped_by_orders["size_of_the_basket"] = grouped_by_orders["product_name"].map(len)
print(grouped_by_orders)

feature_cols = [
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "size_of_the_basket"
]

X = pd.get_dummies(grouped_by_orders[feature_cols])

y = grouped_by_orders["product_name"]

mlb = MultiLabelBinarizer(sparse_output=True)
y_matrice = mlb.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_matrice, test_size=0.2, random_state=42
)

model = OneVsRestClassifier(
    BernoulliNB()
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(y_pred)

         order_id                                       product_name  user_id  \
0               2  [Organic Egg Whites, Michigan Organic Kale, Ca...   202279   
1               4                            [Original Orange Juice]   178520   
2              22  [2% Reduced Fat Milk, Banana, Iceberg Lettuce,...    30822   
3              23  [Organic Baby Spinach Salad, Potato Yukon Gold...    68572   
4              26  [Organic Unsweetened Almond Milk, Banana, Bone...   153404   
...           ...                                                ...      ...   
1112182   2999994      [Bag of Organic Bananas, Organic Blueberries]   184942   
1112183   2999995  [I Heart Baby Kale, Organic Mexican Blend Fine...   128711   
1112184   2999997  [Organic Baby Spinach, Goat Cheese Crumbles, O...    41179   
1112185   2999998  [Organic Low Sodium Chicken Broth, Grated Parm...    68878   
1112186   2999999  [Organic Salted Butter, Organic Baby Spinach, ...   100467   

         order_number  orde

## Modèle 3 : Prédire

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")
df = df.dropna()

next_days_since_order = df['days_since_prior_order'].shift(-1)

X = df[['days_since_prior_order']]
y = next_days_since_order

mask = y.notna()
X = X[mask]
y = y[mask]

model = LinearRegression()



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X,y)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test , y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(mae)
print(rmse)
print(r2)